# <span style="color:purple"> Import Raw Data </span>

## <span style="color:purple"> Find Files </span>

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import sys

# Establish the downloaded AMISOR database as the root from which to find MicroCAT data
root = Path("../data/raw/AMISOR")

# Find all MicroCAT data files 
dat_files = sorted(root.rglob("*.dat"))

# Regex (regular expression): find AM01..AM06 anywhere in path/name + cat#### filename
rx = re.compile(r"(AM\d{2}).*?\bcat(\d+)(?:_[^/]*)?\.dat\b", re.IGNORECASE)

## <span style="color:purple"> Parse Correct MicroCAT Files to Dataframe Objects </span>

### <span style="color:purple"> Parsing Function </span>

<span style="color:purple"> Parses in the following values from each .dat file, saving them as the corresponding variable: </span> \
<span style="color:purple"> t: in-situ temperature, degrees C </span> \
<span style="color:purple"> C: conductivity, mS/cm </span> \
<span style="color:purple"> p: pressure, dbar </span> \
<span style="color:purple"> SP: practical salinity, PSS78 </span>

In [ ]:
# Define a function for parsing MicroCAT data to pandas dataframe object
def parse_microcat_dat(path: Path, skiprows: int = 5) -> pd.DataFrame:
    # Load the file containing data
    data = np.loadtxt(path, skiprows=skiprows)

    # There are several columns, but we are interested in parsing in values related to in situ temperature, conductivity, pressure, salinity, and time 
    t = data[:, 1]
    C = data[:, 2]
    p = data[:, 3]
    SP  = data[:, 4]

    day    = data[:, 5].astype(int)
    month  = data[:, 6].astype(int)
    year   = data[:, 7].astype(int)
    hour   = data[:, 8].astype(int)
    minute = data[:, 9].astype(int)
    second = data[:,10].astype(int)

    # Convert parsed datetime data into a standardised pandas datatime object
    time = pd.to_datetime(
        dict(year=year, month=month, day=day, hour=hour, minute=minute, second=second),
        errors="coerce"
    )

    # Store all parsed data into a dataframe object, using the datetime object as the index
    df = pd.DataFrame({"in_situ_temperature": t, "conductivity": cond, "pressure": pres, "practical_salinity": SP}, index=time).sort_index()
    df.index.name = "time"

    # Replace all null data (~-999) with NaN
    df = df.mask(np.isclose(df, -999.0, atol=1e-6))
    
    # OPTIONAL: Delete lines of data where datetime cannot be parsed properly
    # df = df[~df.index.isna()]
    
    return df


### <span style="color:purple"> Ensure each .dat file is assigned to a dataframe object corresponding to a unique borehole location </span>

<span style="color:purple"> defaultdict(list) creates a dictionary where any new/missing key is automatically assigned to a new, empty list
i.e. the below code iterates over all the recognised .dat files, parsing the MicroCAT data to a unique dataframe object, and then adding each dataframe object to a list
of dataframe objects identified by a key corresponding to the AM and cat number </span>

<span style="color:purple"> It will look something like below:
dfs_list = {
    ("AM01","1969"): [df_part1, df_part2],
    ("AM01","1970"): [df_part1],
    ("AM02","1174"): [df_part1, df_part2, df_part3],
} </span>

<span style="color:purple"> In some instances, there are multiple .dat files for a single MicroCAT, often representing different seasonal data downloads. It is necessary to read in all files though, as while there is overlap, each contains unique timestamped data. </span>

In [4]:

# Since there can be multiple .dat files for a single cat (some observational data downloads from different years have been added to the database in separate files)
# Store dataframe objects in a list, keyed by the borehole and cat id (e.g. AM01, cat0001)
dfs_list = defaultdict(list)          # keyed by (AMxx, cat####)
skipped = []                          # files that don’t match the pattern
failed = []                           # files that match but can’t be parsed

# Iterate over all recognised .dat files
for p in dat_files:
    # Skip files containing certain words
    exclude_words = ["uncorrected", "mean"]
    name_lower = p.name.lower()
    if any(word in name_lower for word in exclude_words):
        continue
    
    # Use regex to identify borehole (AMXX) and cat#### in .dat file
    m = rx.search(str(p))
    if not m:
        skipped.append(p)
        continue

    # Determine the borehole id and cat id from the filename
    am_id = m.group(1).upper()
    cat_id = m.group(2)  # keep as string; cast to int if you prefer

    # Parse the MicroDAT data and assign to a dataframe object
    # Also assign the borehole, cat id, and source filename to the dataframe
    try:
        df = parse_microcat_dat(p, skiprows=5)
        df["AM"] = am_id
        df["cat"] = int(cat_id)
        df["source_file"] = p.name

        # Use the borehole and cat ids as keys to identify the list of dataframe objects
        dfs_list[(am_id, cat_id)].append(df)
    
    except Exception as e:
        failed.append((p, repr(e)))

print(dfs_list.keys())


dict_keys([('AM03', '4054'), ('AM03', '4055'), ('AM05', '7322'), ('AM05', '7326'), ('AM05', '7327'), ('AM06', '7323'), ('AM06', '7324'), ('AM06', '7325'), ('AM01', '1969'), ('AM01', '1970'), ('AM01', '1971'), ('AM02', '1174'), ('AM02', '1623'), ('AM02', '1624'), ('AM03', '3883'), ('AM04', '1972'), ('AM04', '1973'), ('AM04', '1974')])


## <span style="color:purple"> Collate Data into One Dataframe Object </span>

### <span style="color:purple"> Function to remove duplicate timestamps </span>

<span style="color:purple"> Since multiple dataframe objects may represent a given borehole, identical timestamps need to be resolved to ensure there is no overlap in data. </span>

In [ ]:
# Set up a function to resolve different cases for where there are duplicates between data files for a specific timestamp
def resolve_duplicates(df):
    resolved_rows = []
    flagged = []

    # Iterate over each group defined by each timestep => matching timesteps have been grouped together
    for timestamp, group in df.groupby(df.index):

        # If there is a unique timestamp, then there is no duplicate and we are done
        if len(group) == 1:
            resolved_rows.append(group.iloc[0])
            continue

        # Within the duplicate timestamps, we want to only select numerical data types (i.e. t, cond, pres, sal) to compare
        numeric = group.select_dtypes(include=[np.number])

        # Case 1: Data in duplicated timestamps is identical (i.e. there is only one unique set of data values)
        # Arbitrarily take the first row
        if numeric.nunique().max() == 1:
            resolved_rows.append(group.iloc[0])
            continue

        # Case 2: Data in 1+ duplicated timestamps is all -999
        # Identify rows where all numeric columns are -999
        mask_all_bad = numeric.isna().all(axis=1)

        # If at least one good row exists
        if (~mask_all_bad).any():
        
            # Drop all fully-bad rows
            group_clean = group[~mask_all_bad]
            numeric_clean = numeric[~mask_all_bad]
        
            # Case 1: There is remaining duplicated timesteps with identical data
            if numeric_clean.nunique().max() == 1:
                resolved_rows.append(group_clean.iloc[0])
                continue
        
            # Case 3: There is remaining duplicated timesteps with conflicting data
            print("\nConflict at:", timestamp)
            print(group_clean)
            flagged.append((timestamp, group_clean))
        
            resolved_rows.append(group_clean.iloc[0])
            continue

        # Case 3: Different valid values → flag for inspection
        print("\nConflict at:", timestamp)
        print(group)
        flagged.append((timestamp, group))

        # For now, keep first arbitrarily
        resolved_rows.append(group.iloc[0])

    result = pd.DataFrame(resolved_rows)
    result.index.name = "time"

    return result.sort_index(), flagged

### <span style="color:purple"> Concatenate dataframes for each borehole into one dataframe </span>

In [6]:
# Set up a dataframe object in which to concatenate the parsed dataframe objects corresponding to each AM and cat
dfs = {}    # keyed by (AMxx, cat####)
    
# Iterate over list of dataframes (pertaining to individual .dat files) defined by each key in the dictionary, combining each list of dataframes into one dataframe
for key, parts in dfs_list.items():
    # Stack all dataframes in the list corresponding to a key and sort by time => then concatenate into one dataframe
    out = pd.concat(parts).sort_index()

    # Since there is minor overlap in data timestamps between files, we choose to keep the last occurrence here
    out, flagged_conflicts = resolve_duplicates(out)

    dfs[key] = out

print(dfs.keys())

print(f"Matched+parsed: {len(dfs)} | skipped (no AM/cat): {len(skipped)} | failed parse: {len(failed)}")

dict_keys([('AM03', '4054'), ('AM03', '4055'), ('AM05', '7322'), ('AM05', '7326'), ('AM05', '7327'), ('AM06', '7323'), ('AM06', '7324'), ('AM06', '7325'), ('AM01', '1969'), ('AM01', '1970'), ('AM01', '1971'), ('AM02', '1174'), ('AM02', '1623'), ('AM02', '1624'), ('AM03', '3883'), ('AM04', '1972'), ('AM04', '1973'), ('AM04', '1974')])
Matched+parsed: 18 | skipped (no AM/cat): 34 | failed parse: 0


### <span style="color:purple"> Combine all data into one dataframe object </span>

In [7]:
# Combine into one DataFrame
frames = []
for (borehole, cat), df in dfs.items():
    df = df.reset_index()  # make time a column again
    df["borehole"] = borehole
    df["cat"] = cat
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)

## <span style="color:purple"> Handle Known Sensor Errors/Issues </span>

### <span style="color:purple"> AM01: Pressure Sensor Failure </span>

<span style="color:purple"> For AM01, pressure sensor data were bad for all 3 instruments, and a constant pressure value was selected for each instrument (from inspection of data near the start in January 2002), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of AM01 instruments is: </span> \
<span style="color:purple">cat1969 = 441.1 dbar </span> \
<span style="color:purple">cat1970 = 581.5 dbar </span> \
<span style="color:purple">cat1971 = 743.0 dbar </span>

In [ ]:
# Override each pressure for each respective cat with the constant values as above
df_all.loc[(df_all["borehole"] == "AM01") & (df_all["cat"] == "1969"), "pressure"] = 441.1
df_all.loc[(df_all["borehole"] == "AM01") & (df_all["cat"] == "1970"), "pressure"] = 581.5
df_all.loc[(df_all["borehole"] == "AM01") & (df_all["cat"] == "1971"), "pressure"] = 743.0

### <span style="color:purple"> Save Dataframe as Parquet </span>

In [9]:
# Save to parquet file for efficient storage and future use
# Note: parquet is a columnar storage format that is more efficient for large datasets and preserves data types better than CSV
# Note: parquet files can be read back into pandas with pd.read_parquet("data/proc/01_imported.parquet")
# Note: this will overwrite any existing file at this location
df_all.to_parquet("../data/proc/01_imported.parquet", index=False)

### <span style="color:purple"> Check column names of dataframe </span>

In [10]:
df_all.columns  

Index(['time', 'temperature', 'conductivity', 'pressure', 'salinity', 'AM',
       'cat', 'source_file', 'borehole'],
      dtype='object')